# Multi-Fidelity MEGNet Band Gap Model

Reprodução do notebook original do MatGL com ajustes para:
- matgl 1.3.0 (imports atualizados)
- GPU (RTX 4060 Laptop)
- DGL 2.4.0

In [ ]:
from __future__ import annotations

import gzip
import json
import os
import shutil
import warnings
import zipfile
from copy import deepcopy
from functools import partial

import lightning as L
import matplotlib.pyplot as plt
import pandas as pd
import requests
import torch
from dgl.data.utils import split_dataset
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
from lightning.pytorch.loggers import CSVLogger
from pymatgen.core import Structure

from matgl.config import DEFAULT_ELEMENTS
from matgl.ext.pymatgen import Structure2Graph
from matgl.graph.data import MGLDataLoader, MGLDataset, collate_fn_graph
from matgl.models import MEGNet
from matgl.utils.training import ModelLightningModule

warnings.simplefilter("ignore")
import os
import sys
from pathlib import Path

NOTEBOOK_DIR = Path(os.path.abspath('')).resolve()
_candidate_roots = [NOTEBOOK_DIR, *NOTEBOOK_DIR.parents]
FINAL_ROOT = next((p if p.name == 'final' else p / 'final' for p in _candidate_roots if p.name == 'final' or (p / 'final').is_dir()), None)
if FINAL_ROOT is None:
    raise RuntimeError(f'Não foi possível localizar final/ a partir de {NOTEBOOK_DIR}')
FINAL_ROOT = FINAL_ROOT.resolve()
if str(FINAL_ROOT) not in sys.path:
    sys.path.insert(0, str(FINAL_ROOT))

from common import DATA_DIR, FINAL_ROOT, REPO_ROOT, RUNS_DIR, ensure_run_dir, required_path

print(f'Final root: {FINAL_ROOT}')
if torch.cuda.is_available():
    torch.set_float32_matmul_precision('high')


## Dataset

Download do dataset MP.2019.4.1 do figshare. Os dados de bandgap estão em 4 fidelidades:
- **PBE** (0): mais barato, subestima o gap
- **GLLB-SC** (1): corrige gaps de semicondutores
- **HSE** (2): híbrido, mais preciso
- **SCAN** (3): meta-GGA melhorado

In [ ]:
RUN_ID = "000"
RUN_NAME = "megnet_pretrain_mp"
RUN_DIR = ensure_run_dir(RUN_ID, RUN_NAME)
(RUN_DIR / "model" / "best").mkdir(parents=True, exist_ok=True)
(RUN_DIR / "cache").mkdir(parents=True, exist_ok=True)

MP_STRUCT_PATH = required_path(DATA_DIR / "mp.2019.04.01.json", "MP structures")
MP_BGAP_PATH = required_path(DATA_DIR / "band_gap_no_structs.gz", "MP band gaps")

ALL_FIDELITIES = ["pbe", "gllb-sc", "hse", "scan"]
CUTOFF = 5.0
BATCH_SIZE = 64
NUM_WORKERS = 4
LR = 1e-3
WEIGHT_DECAY = 1e-5
MAX_EPOCHS = 180
PATIENCE = 30
SEED = 42
FORCE_RETRAIN = True
ACCELERATOR = "gpu" if torch.cuda.is_available() else "cpu"
DEVICES = 1

torch.manual_seed(SEED)

PRETRAIN_CKPTS = sorted((RUN_DIR / "model" / "best").glob("*.ckpt"))
USE_EXISTING_ARTIFACTS = bool(PRETRAIN_CKPTS) and not FORCE_RETRAIN

if USE_EXISTING_ARTIFACTS:
    print(f"Checkpoint existente encontrado; leitura completa do MP será pulada: {PRETRAIN_CKPTS[-1].name}")
    structure_data = {}
    bandgap_data = {}
    useful_ids = set()
else:
    with open(MP_STRUCT_PATH) as f:
        structure_data = {i["material_id"]: i["structure"] for i in json.load(f)}
    print(f"Total de estruturas no MP.2019.4.1: {len(structure_data)}")

    with gzip.open(MP_BGAP_PATH, "rb") as f:
        bandgap_data = json.loads(f.read())

    useful_ids = set.union(*[set(bandgap_data[i].keys()) for i in ALL_FIDELITIES])
    print(f"Estruturas com pelo menos 1 fidelidade de bandgap: {len(useful_ids)}")

    print("Convertendo CIF → pymatgen Structure (pode demorar alguns minutos)...")
    structure_data = {i: structure_data[i] for i in useful_ids}
    structure_data = {i: Structure.from_str(j, fmt="cif") for i, j in structure_data.items()}
    print("Concluído.")
print(f"Run: {RUN_DIR}")

## Visualização de uma estrutura

Escolha qualquer `mp_id` do dataset para inspecionar a estrutura cristalina em 3D e comparar os valores de bandgap entre as fidelidades disponíveis.

In [ ]:
print("Visualização estrutural removida do pipeline executável; o pré-treino final usa os artefatos em final/runs/000_megnet_pretrain_mp.")

## Geração dos grafos e labels por fidelidade

A fidelidade é codificada no **state attribute** (atributo global do grafo):
- O mesmo material aparece múltiplas vezes, uma por fidelidade disponível
- O modelo aprende a mapear (estrutura, fidelidade) → bandgap

In [ ]:
if USE_EXISTING_ARTIFACTS:
    print("Célula pulada: artefatos de pré-treino já disponíveis em final/runs/000_megnet_pretrain_mp.")
else:
    structures = []
    material_ids = []
    graph_attrs = []
    targets = []

    for fidelity_id, fidelity in enumerate(ALL_FIDELITIES):
        for mp_id in bandgap_data[fidelity]:
            structure = deepcopy(structure_data[mp_id])
            graph_attrs.append(fidelity_id)   # PBE=0, GLLB-SC=1, HSE=2, SCAN=3
            structures.append(structure)
            targets.append(bandgap_data[fidelity][mp_id])
            material_ids.append(f"{mp_id}_{fidelity}")

    print(f"Total de amostras (estrutura × fidelidade): {len(structures)}")
    for fid_id, fid in enumerate(ALL_FIDELITIES):
        count = graph_attrs.count(fid_id)
        print(f"  {fid.upper():8s}: {count}")

## Configuração do Dataset

In [ ]:
if USE_EXISTING_ARTIFACTS:
    print("Célula pulada: artefatos de pré-treino já disponíveis em final/runs/000_megnet_pretrain_mp.")
else:
    element_types = DEFAULT_ELEMENTS
    cry_graph = Structure2Graph(element_types=element_types, cutoff=CUTOFF)

    labels = {"bandgap": targets}
    dataset = MGLDataset(
        structures=structures,
        graph_labels=graph_attrs,
        labels=labels,
        converter=cry_graph,
        filename=str(RUN_DIR / "cache" / "MP_MGLDataset"),
    )
    print(f"Dataset criado com {len(dataset)} amostras")

## Split train/val/test

In [ ]:
if USE_EXISTING_ARTIFACTS:
    print("Célula pulada: artefatos de pré-treino já disponíveis em final/runs/000_megnet_pretrain_mp.")
else:
    # 80% train, 10% val, 10% test
    train_data, val_data, test_data = split_dataset(
        dataset,
        frac_list=[0.8, 0.1, 0.1],
        shuffle=True,
        random_state=SEED,
    )
    print(f"Train: {len(train_data)} | Val: {len(val_data)} | Test: {len(test_data)}")

    my_collate_fn = partial(collate_fn_graph, include_line_graph=False)

    train_loader, val_loader, test_loader = MGLDataLoader(
        train_data=train_data,
        val_data=val_data,
        test_data=test_data,
        collate_fn=my_collate_fn,
        batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    )

## Modelo MEGNet

Parâmetros chave:
- `ntypes_state=4`: um embedding por fidelidade (PBE, GLLB-SC, HSE, SCAN)
- `dim_state_embedding=64`: dimensão do vetor de estado
- `readout_type="set2set"`: agregação permutation-invariant
- `is_intensive=True`: predição de propriedade intensiva (por estrutura, não por átomo)

In [ ]:
if USE_EXISTING_ARTIFACTS:
    print("Célula pulada: artefatos de pré-treino já disponíveis em final/runs/000_megnet_pretrain_mp.")
else:
    model = MEGNet(
        element_types=element_types,
        cutoff=CUTOFF,
        is_intensive=True,
        dim_state_embedding=64,
        ntypes_state=4,
        readout_type="set2set",
        include_states=True,
    )

    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    lit_module = ModelLightningModule(model=model, optimizer=optimizer, lr=LR)
    print(f"Parâmetros treináveis: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## Treinamento (GPU)

In [ ]:
if USE_EXISTING_ARTIFACTS:
    print("Célula pulada: artefatos de pré-treino já disponíveis em final/runs/000_megnet_pretrain_mp.")
else:
    best_dir = RUN_DIR / "model" / "best"
    existing_ckpts = sorted(best_dir.glob("*.ckpt"))

    if existing_ckpts and not FORCE_RETRAIN:
        ckpt_path = existing_ckpts[-1]
        print(f"Checkpoint existente encontrado; pulando treino: {ckpt_path.name}")
        lit_module = ModelLightningModule.load_from_checkpoint(str(ckpt_path), model=model, map_location="cpu")
        model = lit_module.model
        trainer = L.Trainer(accelerator=ACCELERATOR, devices=DEVICES, logger=False, enable_progress_bar=False)
    else:
        logger = CSVLogger(str(RUN_DIR), name="logs")
        trainer = L.Trainer(
            max_epochs=MAX_EPOCHS,
            accelerator=ACCELERATOR,
            devices=DEVICES,
            logger=logger,
        log_every_n_steps=20,
            callbacks=[
                EarlyStopping(monitor="val_MAE", patience=PATIENCE, mode="min", verbose=True),
                ModelCheckpoint(
                    dirpath=str(best_dir),
                    filename="best-{epoch:03d}-{val_MAE:.4f}",
                    monitor="val_MAE",
                    mode="min",
                    save_top_k=1,
                ),
            ],
        )
        trainer.fit(model=lit_module, train_dataloaders=train_loader, val_dataloaders=val_loader)

## Convergência

In [ ]:
import glob

metrics_files = sorted(glob.glob(str(RUN_DIR / "logs" / "version_*" / "metrics.csv")))
if not metrics_files:
    metrics_files = sorted(glob.glob(str(RUN_DIR / "version_*" / "metrics.csv")))

if metrics_files:
    metrics = pd.read_csv(metrics_files[-1])
    fig, ax = plt.subplots(figsize=(8, 4))
    metrics["train_MAE"].dropna().reset_index(drop=True).plot(ax=ax, label="Train MAE")
    metrics["val_MAE"].dropna().reset_index(drop=True).plot(ax=ax, label="Val MAE")
    ax.set_xlabel("Época")
    ax.set_ylabel("MAE (eV)")
    ax.set_title("Convergência - MEGNet Multi-Fidelity Band Gap")
    ax.legend()
    plt.tight_layout()
    plt.savefig(RUN_DIR / "figures" / "convergence.png", dpi=150)
    plt.show()
else:
    print("Nenhum metrics.csv encontrado; figura de convergência não gerada.")

## Avaliação no conjunto de teste

In [ ]:
if USE_EXISTING_ARTIFACTS:
    print("Célula pulada: artefatos de pré-treino já disponíveis em final/runs/000_megnet_pretrain_mp.")
else:
    trainer.test(model=lit_module, dataloaders=test_loader)

In [ ]:
if USE_EXISTING_ARTIFACTS:
    print("Célula pulada: artefatos de pré-treino já disponíveis em final/runs/000_megnet_pretrain_mp.")
else:
    import random

    FIDELITY_NAMES = {0: "PBE", 1: "GLLB-SC", 2: "HSE", 3: "SCAN"}
    model.eval()
    random.seed(SEED)

    sample_indices = random.sample(range(len(test_data)), min(5, len(test_data)))
    rows = []
    for idx in sample_indices:
        _, _, state, labels = test_data[idx]
        true_val = labels["bandgap"].item()
        fidelity_id = int(state)
        orig_idx = test_data.indices[idx]
        mid = material_ids[orig_idx]
        mp_id = mid.rsplit("_", 1)[0]
        struct = structure_data[mp_id]
        state_t = torch.tensor([fidelity_id], dtype=torch.long)
        pred_val = model.predict_structure(struct, state_attr=state_t).item()
        rows.append({
            "material_id": mp_id,
            "formula": struct.composition.reduced_formula,
            "fidelity": FIDELITY_NAMES.get(fidelity_id, str(fidelity_id)),
            "gap_true": true_val,
            "gap_pred": pred_val,
            "abs_error": abs(pred_val - true_val),
        })

    sample_df = pd.DataFrame(rows)
    sample_df.to_csv(RUN_DIR / "outputs" / "sample_predictions.csv", index=False)
    display(sample_df)

In [ ]:
if USE_EXISTING_ARTIFACTS:
    print("Célula pulada: artefatos de pré-treino já disponíveis em final/runs/000_megnet_pretrain_mp.")
else:
    print("Visualizações 3D interativas foram removidas do pipeline executável final; os exemplos numéricos foram salvos em outputs/sample_predictions.csv.")

## Limpeza (opcional - remova se quiser manter os arquivos)

In [ ]:
print(f"Artefatos mantidos em: {RUN_DIR.resolve()}")